# Etapa 15 · Recuperación híbrida sobre las 34 fichas de Memoria Chilena

**Ejecuta solamente este notebook** desde `notebooks/` mediante **Run → Run All Cells**. No debes mover ningún ZIP, repetir descargas ni configurar una API.

Este notebook lee automáticamente `data/input/etapa14.zip` (incluido en el proyecto) y reconstruye **desde cero** los embeddings del catálogo ampliado. Compara cuatro métodos de recuperación:

- **A:** TF-IDF de palabras y caracteres.
- **B:** embeddings semánticos multilingües MiniLM.
- **C:** fusión léxico + semántico (RRF, *Reciprocal Rank Fusion*).
- **D:** método C más una expansión **experimental** con relaciones catalográficas específicas.

Produce `data/output/etapa15.zip` automáticamente. Los documentos se muestran con enlaces a las fichas originales y **no** se generan afirmaciones históricas desde PDF u OCR. Las etiquetas de minisitio son datos curatoriales, **no hechos históricos**.

El modelo MiniLM se descarga desde Hugging Face en la primera ejecución, si aún no está guardado en la caché local.

## 0. Dependencias

Si falta una biblioteca, esta primera celda la instala en el mismo entorno de Python que usa Jupyter. No instala Tesseract, no llama a Groq y no descarga PDFs.

In [1]:
import importlib.util, subprocess, sys
PAQUETES = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.0,<3",
    "sklearn": "scikit-learn>=1.3",
    "sentence_transformers": "sentence-transformers>=3.0,<7",
}
faltan = [pip for modulo, pip in PAQUETES.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando bibliotecas faltantes:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltan])
else:
    print("Dependencias ya instaladas.")

Dependencias ya instaladas.


## 1. Abrir el ZIP de la etapa 14 y validar el catálogo

El archivo se busca en `data/output/` y `data/input/`. Si ejecutas el notebook desde otra carpeta, también localiza la raíz del proyecto hacia arriba.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from itertools import combinations
from hashlib import sha256
import io, json, math, re, unicodedata, zipfile
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CWD = Path.cwd().resolve()
candidatas = [CWD, *CWD.parents]
ROOT = next((p for p in candidatas if (p/"notebooks").is_dir() and (p/"data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Abre Jupyter desde la carpeta del proyecto memoria_chilena o desde notebooks/.")
OUT = ROOT / "data" / "output"
OUT.mkdir(parents=True, exist_ok=True)
RUTA_ZIP = next((p for p in [OUT/"etapa14.zip",ROOT/"data"/"input"/"etapa14.zip"] if p.is_file()), None)
if RUTA_ZIP is None:
    raise FileNotFoundError("No encuentro etapa14.zip en data/output o data/input. Usa el proyecto completo de etapa 15.")
with zipfile.ZipFile(RUTA_ZIP) as z:
    CATALOGO = pd.read_csv(io.BytesIO(z.read("catalogo_ampliado.csv")), dtype=str, keep_default_na=False)
    MINISITIOS = pd.read_csv(io.BytesIO(z.read("minisitios_auditados.csv")), dtype=str, keep_default_na=False)
    INFORME_14 = json.loads(z.read("informe_etapa14.json").decode("utf-8"))
    HASH_ZIP = sha256(RUTA_ZIP.read_bytes()).hexdigest()

REQUERIDAS = {'mcid','ficha','titulo','autor','fecha','materias','unidades','notas','datos','minisitios','origen'}
if not REQUERIDAS.issubset(CATALOGO.columns):
    raise ValueError(f"Faltan columnas requeridas: {sorted(REQUERIDAS-set(CATALOGO.columns))}")
if CATALOGO['ficha'].duplicated().any() or CATALOGO['mcid'].duplicated().any():
    raise ValueError('Hay identificadores o fichas duplicados: no continuar sin auditar el catálogo.')
if not CATALOGO['ficha'].str.startswith('https://www.memoriachilena.gob.cl/').all():
    raise ValueError('Hay URL fuera del dominio de Memoria Chilena; revisa el CSV de entrada.')
CATALOGO = CATALOGO.sort_values('mcid').reset_index(drop=True)
IDS = CATALOGO['mcid'].tolist()
N = len(IDS)
print('Proyecto:', ROOT)
print('ZIP de entrada:', RUTA_ZIP)
print('Fichas:', N, '| nuevas:', (CATALOGO.origen=='etapa14').sum(), '| sin título:', (CATALOGO.titulo.str.strip()=='').sum())
print('Triples nuevos (según etapa14):', INFORME_14.get('triples_originales_nuevos'))
print('Autores presentes:', int((CATALOGO.autor.str.strip()!='').sum()), '| notas presentes:',int((CATALOGO.notas.str.strip()!='').sum()))

Proyecto: C:\Users\jvera\Documents\memoria_chilena
ZIP de entrada: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa14.zip
Fichas: 34 | nuevas: 24 | sin título: 0
Triples nuevos (según etapa14): 4082
Autores presentes: 30 | notas presentes: 14


## 2. Preparar representaciones textuales

Conservamos los títulos, grafías originales y fechas tal como fueron catalogados. La normalización léxica permite buscar variantes como *sufrajio/sufragio* o *Pirú/Perú*, pero **no altera los archivos fuente**. La representación semántica se forma con título, autor, materias, nota y nombre legible del minisitio (no URL).

In [3]:
MAPA_MINISITIOS={
    'https://www.memoriachilena.gob.cl/602/w3-article-790.html':'Religiosidad andina',
}
MAPA_MINISITIOS.update(dict(zip(MINISITIOS.url, MINISITIOS.minisitio)))

def grupos_legibles(row):
    return ' | '.join(MAPA_MINISITIOS.get(s.strip(),s.strip()) for s in row['minisitios'].split(' | ') if s.strip())

def normalizar(texto):
    s=unicodedata.normalize('NFKD',str(texto).lower())
    s=''.join(c for c in s if not unicodedata.combining(c))
    s=re.sub(r'[^a-z0-9\s]', ' ', s)
    cambios={'sufrajio':'sufragio','quichua':'quechua','piru':'peru','yncas':'incas'}
    palabras=[cambios.get(w,w) for w in s.split()]
    return ' '.join(palabras)

def representacion(row):
    piezas=[f"Título: {row.titulo}",f"Autor: {row.autor}",
            f"Materias: {row.materias.replace('%20',' ')}",
            f"Colección temática: {row.colecciones_legibles}",
            f"Nota catalográfica: {row.notas[:500]}"]
    return '. '.join(p for p in piezas if p.split(': ',1)[-1].strip())

CATALOGO['colecciones_legibles'] = CATALOGO.apply(grupos_legibles,axis=1)
CATALOGO['texto_semantico'] = CATALOGO.apply(representacion,axis=1)
CATALOGO['texto_lexico'] = CATALOGO.apply(lambda r:
    ' '.join([r.titulo]*4+[r.autor]*2+[r.materias.replace('%20',' ')]*2+
             [r.colecciones_legibles]+[r.notas[:500]]), axis=1)

V_PAL = TfidfVectorizer(preprocessor=normalizar, ngram_range=(1,2),min_df=1,sublinear_tf=True)
V_CHAR = TfidfVectorizer(preprocessor=normalizar, analyzer='char_wb',ngram_range=(3,5), min_df=1,sublinear_tf=True)
X_PAL=V_PAL.fit_transform(CATALOGO.texto_lexico.tolist())
X_CHAR=V_CHAR.fit_transform(CATALOGO.texto_lexico.tolist())
TEXT_HASH = sha256(('\n'.join(f'{id_}\t{s}' for id_,s in zip(IDS,CATALOGO.texto_semantico))).encode('utf-8')).hexdigest()
print('Documentos:',len(IDS),'| vocabulario palabras:',len(V_PAL.vocabulary_),'| vocabulario caracteres:',len(V_CHAR.vocabulary_))
print('Corpus semántico SHA256:',TEXT_HASH[:16])

Documentos: 34 | vocabulario palabras: 1013 | vocabulario caracteres: 3941
Corpus semántico SHA256: a2055a918d68b298


## 3. Generar embeddings MiniLM auténticos (384 dimensiones)

Si el modelo ya está descargado, se carga desde la caché de Hugging Face. Se guardan los vectores y un hash del corpus en `data/output/etapa15/`. Si el catálogo cambia, se regeneran, sin mezclar embeddings anteriores.

In [4]:
from sentence_transformers import SentenceTransformer
MODELO_ID='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
DESTINO = OUT/'etapa15'
DESTINO.mkdir(parents=True,exist_ok=True)
CACHE=DESTINO/'embeddings_34.npz'
MODELO=SentenceTransformer(MODELO_ID,device='cpu')

cargado=False
if CACHE.is_file():
    try:
        with np.load(CACHE,allow_pickle=False) as c:
            correcto=(str(c['corpus_sha256'].item())==TEXT_HASH and
                      str(c['model_id'].item())==MODELO_ID and
                      c['ids'].tolist()==IDS)
            if correcto:
                E=np.asarray(c['vectors'],dtype='float32')
                cargado=(E.shape==(N,384) and np.isfinite(E).all())
    except (ValueError,KeyError,EOFError):
        cargado=False
if not cargado:
    E=MODELO.encode(CATALOGO.texto_semantico.tolist(),batch_size=16,show_progress_bar=True,
                    convert_to_numpy=True,normalize_embeddings=True).astype('float32')
    if E.shape!=(N,384) or not np.isfinite(E).all():
        raise ValueError(f'Embeddings inválidos, shape={E.shape}; no generar rankings.')
    np.savez_compressed(CACHE,vectors=E,ids=np.asarray(IDS),corpus_sha256=np.array(TEXT_HASH),model_id=np.array(MODELO_ID))
if np.min(np.linalg.norm(E,axis=1))<0.99:
    raise ValueError('Los embeddings deben estar normalizados; comprobar el modelo.')
print('MiniLM real:',E.shape,'| caché reutilizada:',cargado)

def scores_consulta(pregunta):
    a_pal=cosine_similarity(V_PAL.transform([pregunta]),X_PAL).ravel()
    a_char=cosine_similarity(V_CHAR.transform([pregunta]),X_CHAR).ravel()
    lexical=(0.7*a_pal+0.3*a_char).astype(float)
    q=np.asarray(MODELO.encode([pregunta],convert_to_numpy=True,normalize_embeddings=True),dtype='float32')[0]
    semantico=(E @ q).astype(float)
    return lexical,semantico

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

MiniLM real: (34, 384) | caché reutilizada: False


## 4. Relaciones específicas del catálogo: grafo controlado

Excluimos de la **expansión** los cuatro nombres generales de minisitio, para evitar redes completas por co-pertenencia. Las relaciones permitidas surgen únicamente de **unidades temáticas compartidas y poco frecuentes** (2–5 fichas), no de contenido histórico inferido. La conexión y su etiqueta quedan auditadas.

In [5]:
UNIDADES=[set(x.strip() for x in s.split('|') if x.strip()) for s in CATALOGO.unidades]
FRECUENCIAS=Counter(t for fila in UNIDADES for t in fila)
GENERICAS={
 'religiosidadandina',
 'elsufragiodelasmujeresenchile(1865-1952)',
 'memoriaddeformaciondelanacion(1810-1879)',
 'lasfiestasdelcentenario:celebracionycritica,1910',
}
MATRIZ=np.zeros((N,N),dtype=float)
ARISTAS=[]
for i,j in combinations(range(N),2):
    especificas=sorted(t for t in UNIDADES[i]&UNIDADES[j] if t not in GENERICAS and 2<=FRECUENCIAS[t]<=5)
    if not especificas: continue
    peso=sum(math.log((N+1)/(FRECUENCIAS[t]+1)) for t in especificas)
    MATRIZ[i,j]=MATRIZ[j,i]=peso
    ARISTAS.append({'mcid_1':IDS[i], 'mcid_2':IDS[j], 'etiquetas': ' | '.join(especificas), 'peso': round(peso,6), 'tipo':'coincidencia_curatorial_especifica'})
ARISTAS_DF=pd.DataFrame(ARISTAS,columns=['mcid_1','mcid_2','etiquetas','peso','tipo'])
N_GRADOS=np.count_nonzero(MATRIZ,axis=0)
print('Relaciones específicas:',len(ARISTAS_DF),'| documentos con al menos una relación:',int((N_GRADOS>0).sum()),'| aislados:',int((N_GRADOS==0).sum()))
print('Ninguna arista representa una relación histórica causal.')

Relaciones específicas: 23 | documentos con al menos una relación: 17 | aislados: 17
Ninguna arista representa una relación histórica causal.


## 5. Comparación de los cuatro modelos

Para fusionar rankings empleamos **RRF (k=60)**, no una suma directa de TF-IDF y coseno. Para el método D añadimos una bonificación pequeña y acotada de vecinos temáticos de los tres primeros resultados; **el grafo no reemplaza al recuperador**. Este peso es un hiperparámetro provisional que deberá evaluarse con juicios humanos.

In [6]:
METODOS=('A_tfidf','B_embeddings','C_hibrido_rrf','D_hibrido_rdf')

def orden(scores):
    return sorted(range(N),key=lambda i:(-float(scores[i]),IDS[i]))

def cuatro_rankings(pregunta):
    lex,sem=scores_consulta(pregunta)
    ordA,ordB=orden(lex),orden(sem)
    rankA={i:r+1 for r,i in enumerate(ordA)}
    rankB={i:r+1 for r,i in enumerate(ordB)}
    rr=np.array([(1/(60+rankB[i]) + (1/(60+rankA[i]) if lex[i]>0 else 0.0)) for i in range(N)],dtype=float)
    ordC=orden(rr)
    # Una expansión local y acotada, solo relaciones de tema compartido verificado.
    bono=np.zeros(N,dtype=float)
    for i in ordC[:3]:
        vec=MATRIZ[i].copy()
        if not np.any(vec>0):continue
        for j in orden(vec)[:3]:
            if vec[j]<=0:continue
            bono[j]+=0.06*rr[i]*(vec[j]/vec.max())
    d=rr+bono
    return {'A_tfidf':lex,'B_embeddings':sem,'C_hibrido_rrf':rr,'D_hibrido_rdf':d}, {'bono_grafo':bono}

CONSULTAS=[
 ('q01','¿Qué obras tratan el voto de las mujeres en Chile?','sufragio'),
 ('q02','¿Qué documentos se relacionan con la ley electoral chilena de 1949?','sufragio'),
 ('q03','¿Qué textos están asociados con Martina Barros de Orrego?','sufragio'),
 ('q04','¿Qué documentos hablan del sufragio femenino y la participación política?','sufragio'),
 ('q05','¿Qué fuentes documentan los inicios del Instituto Nacional de Chile?','formacion_nacion'),
 ('q06','¿Qué escritos de Jotabeche están incluidos en el catálogo?','formacion_nacion'),
 ('q07','¿Qué publicaciones se vinculan al 18 de septiembre de 1810?','formacion_nacion'),
 ('q08','¿Qué libros permiten investigar las celebraciones del centenario de 1910?','centenario'),
 ('q09','¿Qué números de la revista Zig-Zag están vinculados a 1910?','centenario'),
 ('q10','¿Qué textos se vinculan con la situación económica de Chile hacia 1910?','centenario'),
 ('q11','¿Qué documentos coloniales tratan sobre la lengua aymara?','andino'),
 ('q12','¿Qué textos se vinculan con catecismos y evangelización indígena en los Andes?','andino'),
]
RANKS=[]
for qid, pregunta, tema in CONSULTAS:
    scores,_=cuatro_rankings(pregunta)
    for metodo,sc in scores.items():
        for pos,i in enumerate(orden(sc),start=1):
            r=CATALOGO.iloc[i]
            RANKS.append({'query_id':qid,'pregunta':pregunta,'tema':tema,
                          'metodo':metodo,'posicion':pos,'mcid':IDS[i],
                          'titulo':r.titulo,'ficha':r.ficha,'score':float(sc[i]),
                          'colecciones':r.colecciones_legibles})
RANKINGS=pd.DataFrame(RANKS)
assert len(RANKINGS)==len(CONSULTAS)*N*len(METODOS)
print('Rankings:',len(RANKINGS),'| preguntas:',len(CONSULTAS),'| documentos por método:',N)

# Informe de cambios de ranking, sin interpretar como mejora de calidad.
CAMBIOS=[]
for qid,pregunta,tema in CONSULTAS:
    grupos={m:RANKINGS.query('query_id == @qid and metodo == @m').sort_values('posicion').mcid.tolist() for m in METODOS}
    base=grupos['A_tfidf']
    for m in METODOS[1:]:
        b=grupos[m]
        CAMBIOS.append({'query_id':qid,'tema':tema,'metodo_comparado':m,
                        'cambio_top1':int(base[0]!=b[0]),
                        'cambio_integrantes_top5':int(set(base[:5])!=set(b[:5])),
                        'cambio_orden_top5':int(base[:5]!=b[:5])})
CAMBIOS_DF=pd.DataFrame(CAMBIOS)
print(CAMBIOS_DF.groupby('metodo_comparado')[['cambio_top1','cambio_integrantes_top5','cambio_orden_top5']].sum().to_string())

Rankings: 1632 | preguntas: 12 | documentos por método: 34
                  cambio_top1  cambio_integrantes_top5  cambio_orden_top5
metodo_comparado                                                         
B_embeddings                7                       12                 12
C_hibrido_rrf               4                       12                 12
D_hibrido_rdf               5                       12                 12


## 6. Consultar el catálogo con lenguaje natural

Puedes cambiar la pregunta y el método. Los resultados son **fichas candidatas**, no respuestas históricas generadas.

In [7]:
from IPython.display import display
def buscar(pregunta, metodo='C_hibrido_rrf', top_k=5):
    if metodo not in METODOS:
        raise ValueError('Método inválido: '+str(metodo))
    scores,_=cuatro_rankings(pregunta)
    filas=[]
    for posicion,i in enumerate(orden(scores[metodo])[:top_k],start=1):
        r=CATALOGO.iloc[i]
        filas.append({'posicion':posicion,'mcid':IDS[i],'titulo':r.titulo,'autor':r.autor,
                      'coleccion':r.colecciones_legibles,'ficha':r.ficha,'score':round(float(scores[metodo][i]),6)})
    return pd.DataFrame(filas)

print('Ejemplo: sufragio femenino')
display(buscar('¿Qué publicaciones se refieren al voto femenino en Chile?',top_k=5))
print('Ejemplo: evangelización en los Andes')
display(buscar('¿Qué documentos hablan sobre catecismo en quechua y aymara?',top_k=5))

Ejemplo: sufragio femenino


,posicion,mcid,titulo,autor,coleccion,ficha,score
0,1,MC0061013,El voto femenino,"Barros de Orrego, Martina, 1850-1944",Sufragio femenino (1865–1952),https://www.memoriachilena.gob.cl/602/w3-artic...,0.032018
1,2,MC0031311,Crónica del sufragio femenino en Chile,"Eltit, Diamela, 1949-",Sufragio femenino (1865–1952),https://www.memoriachilena.gob.cl/602/w3-artic...,0.032002
2,3,MC0077873,Ley de Elecciones (Texto definitivo),Chile. Presidencia de la República,Sufragio femenino (1865–1952),https://www.memoriachilena.gob.cl/602/w3-artic...,0.032002
3,4,MC0077872,Ley general sobre inscripciones electorales,Chile. Congreso Nacional,Sufragio femenino (1865–1952),https://www.memoriachilena.gob.cl/602/w3-artic...,0.031099
4,5,MC0008961,Investigaciones sobre la influencia social de ...,"Lastarria, José Victorino, 1817-1888",Memorias de formación de la nación (1810–1879),https://www.memoriachilena.gob.cl/602/w3-artic...,0.030310


Ejemplo: evangelización en los Andes


,posicion,mcid,titulo,autor,coleccion,ficha,score
0,1,MC0014753,Doctrina christiana y catecismo para la instru...,,Religiosidad andina,https://www.memoriachilena.gob.cl/602/w3-artic...,0.032787
1,2,MC0014761,Vocabulario de la lengua aymara,"Bertonio, Ludovico",Religiosidad andina,https://www.memoriachilena.gob.cl/602/w3-artic...,0.031054
2,3,MC0014754,Parte primera de la Chrónica del Perú : que tr...,"Cieza de León, Pedro de, ca. 1520-1554",Religiosidad andina,https://www.memoriachilena.gob.cl/602/w3-artic...,0.030310
3,4,MC0014755,"Primera parte de los commentarios reales, que ...","Vega, Garcilaso de la",Religiosidad andina,https://www.memoriachilena.gob.cl/602/w3-artic...,0.030303
4,5,MC0008961,Investigaciones sobre la influencia social de ...,"Lastarria, José Victorino, 1817-1888",Memorias de formación de la nación (1810–1879),https://www.memoriachilena.gob.cl/602/w3-artic...,0.029514


## 7. Exportar resultados, trazabilidad y plantilla de evaluación

La etapa 15 exporta automáticamente el índice, las 23 relaciones específicas y los rankings. También crea una plantilla **sin calificaciones inventadas**. No se calculan métricas de relevancia a partir de ejemplos ni de etiquetas vacías.

In [8]:
# Entradas documentadas para revisar manualmente la relevancia: unión del top-5 de cuatro métodos.
PLANTILLA=[]
for qid,pregunta,tema in CONSULTAS:
    subconjunto=RANKINGS[(RANKINGS.query_id==qid)&(RANKINGS.posicion<=5)]
    for mcid in sorted(subconjunto.mcid.unique()):
        doc=CATALOGO.loc[CATALOGO.mcid==mcid].iloc[0]
        PLANTILLA.append({'query_id':qid,'pregunta':pregunta,'mcid':mcid,
                          'titulo':doc.titulo,'ficha':doc.ficha,'relevancia_0_1_2':''})
EVALUACION=pd.DataFrame(PLANTILLA)
EVALUACION.to_csv(DESTINO/'plantilla_juicios_pooled.csv',index=False,encoding='utf-8-sig')
CATALOGO.to_csv(DESTINO/'catalogo_indexado.csv',index=False,encoding='utf-8-sig')
ARISTAS_DF.to_csv(DESTINO/'relaciones_especificas.csv',index=False,encoding='utf-8-sig')
RANKINGS.to_csv(DESTINO/'rankings_4_metodos.csv',index=False,encoding='utf-8-sig')
CAMBIOS_DF.to_csv(DESTINO/'cambios_ranking.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(CONSULTAS,columns=['query_id','pregunta','tema']).to_csv(DESTINO/'preguntas_evaluacion.csv',index=False,encoding='utf-8-sig')
REPORTE={
    'etapa':15,'fecha_utc':datetime.now(timezone.utc).isoformat(),
    'entrada':'etapa14.zip','entrada_sha256':HASH_ZIP,
    'fichas':N,'fichas_nuevas_etapa14':int((CATALOGO.origen=='etapa14').sum()),
    'modelo_embedd':MODELO_ID,'dimensiones_embeddings':int(E.shape[1]),
    'indice_generado_sobre_34':bool(N==34),
    'hash_textos_semanticos':TEXT_HASH,
    'preguntas':len(CONSULTAS),'metodos':list(METODOS),
    'registros_ranking':len(RANKINGS),
    'relaciones_curatoriales_especificas':len(ARISTAS_DF),
    'documentos_con_conexion':int((N_GRADOS>0).sum()),
    'documentos_aislados':int((N_GRADOS==0).sum()),
    'juicios_humanos_requeridos':len(EVALUACION),
    'juicios_humanos_completados':0,
    'metricas_relevancia':'no calculadas: no hay etiquetas humanas',
    'observaciones':['Las conexiones del grafo son curatoriales; no hechos históricos.','Las fechas deben interpretarse según el contexto catalográfico.','Los cambios de ranking no prueban mejoras en la relevancia.','Los juicios pooled no miden exhaustivamente el recall del corpus.'],
}
(DESTINO/'informe_etapa15.json').write_text(json.dumps(REPORTE,indent=2,ensure_ascii=False),encoding='utf-8')
(DESTINO/'LEER_RESULTADOS.md').write_text(
    '# Etapa 15: resultados\n\n'
    f'- Documentos indexados: {N}\n- Preguntas: {len(CONSULTAS)}\n'
    f'- Rankings exportados: {len(RANKINGS)}\n- Relaciones curatoriales específicas: {len(ARISTAS_DF)}\n'
    f'- Pares pregunta-documento para calificar en la plantilla: {len(EVALUACION)}\n\n'
    'No usar estas conexiones como prueba de hechos históricos. Sin juicios humanos no existe una comparación formal de calidad.\n',encoding='utf-8')
ARCHIVO=OUT/'etapa15.zip'
with zipfile.ZipFile(ARCHIVO,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DESTINO.iterdir()):
        if f.is_file():
            z.write(f,arcname=f.name)
print('✅ Archivo generado:',ARCHIVO)
print('📋 Candidatos para evaluación humana:',len(EVALUACION))
print('Nota: no se ha consultado Groq, ni descargado PDFs, ni ejecutado OCR.')

✅ Archivo generado: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa15.zip
📋 Candidatos para evaluación humana: 104
Nota: no se ha consultado Groq, ni descargado PDFs, ni ejecutado OCR.


## 8. Próximo paso

1. Revisar los resultados en las nuevas temáticas. Contrastar ranking A/B/C/D, no solo cambios de posición.
2. Calificar los candidatos de `plantilla_juicios_pooled.csv` con relevancia **0 = no relevante, 1 = parcialmente relevante, 2 = relevante**. Estos son juicios de un **pool de candidatos**, no un relevamiento exhaustivo de los 34 documentos.
3. Después evaluaremos la calidad de las respuestas de Qwen/Groq con citas al catálogo y distinción explícita entre evidencia e inferencia.

**Archivo que debes compartir:** `data/output/etapa15.zip`. No envíes tu clave de Groq.